# Assignment 6: Data Aggregation - sales performance by region

Group the Superstore sales by region and product category, find the top-performing region, and compare regions with bar, pie and stacked/grouped bar charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
URL = 'https://raw.githubusercontent.com/Technology-Hatchery/Rcode/master/Data/Sample%20-%20Superstore%20Sales%20(Excel).csv'
sales = pd.read_csv(URL, encoding='latin1')
sales.columns = sales.columns.str.strip()
print('Rows and columns:', sales.shape)
display(sales.head())

## Explore the data and identify useful columns

In [ ]:
print('Columns:', sales.columns.tolist())
display(sales.dtypes.to_frame('data_type'))
display(sales.describe(include='all').T)
print('Missing values:')
display(sales.isna().sum().to_frame('count'))

## Prepare region, category, and sales fields

In [ ]:
# Match common names even if the file uses slightly different capitalization.
def find_column(columns, options):
    lookup = {c.lower().strip(): c for c in columns}
    for option in options:
        if option.lower() in lookup:
            return lookup[option.lower()]
    return None

REGION = find_column(sales.columns, ['Region'])
CATEGORY = find_column(sales.columns, ['Category', 'Product Category'])
AMOUNT = find_column(sales.columns, ['Sales', 'Sales Amount', 'Revenue'])
if not all([REGION, CATEGORY, AMOUNT]):
    raise ValueError(f'Could not find required fields. Available columns: {list(sales.columns)}')

sales[AMOUNT] = pd.to_numeric(sales[AMOUNT], errors='coerce')
sales = sales.dropna(subset=[REGION, CATEGORY, AMOUNT])
print('Using:', {'region': REGION, 'product category': CATEGORY, 'sales amount': AMOUNT})

## Total sales by region and top-performing regions

In [ ]:
region_sales = sales.groupby(REGION, as_index=False)[AMOUNT].sum().sort_values(AMOUNT, ascending=False)
region_sales = region_sales.rename(columns={AMOUNT: 'TotalSales'})
display(region_sales)
print('Top-performing region(s):')
display(region_sales[region_sales['TotalSales'] == region_sales['TotalSales'].max()])

## Visualize sales distribution by region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(region_sales[REGION], region_sales['TotalSales'], color='steelblue')
axes[0].set_title('Total sales by region')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total sales')
axes[0].tick_params(axis='x', rotation=30)
axes[1].pie(region_sales['TotalSales'], labels=region_sales[REGION], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Share of sales by region')
plt.tight_layout()
plt.show()

## Sales by region and product category

In [ ]:
region_category = sales.pivot_table(index=REGION, columns=CATEGORY, values=AMOUNT, aggfunc='sum')
display(region_category)
region_category.plot(kind='bar', stacked=True, figsize=(11, 6), colormap='tab20')
plt.title('Sales by region and product category')
plt.xlabel('Region')
plt.ylabel('Total sales')
plt.xticks(rotation=30)
plt.legend(title='Product category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Grouped bar chart comparison

In [ ]:
region_category.plot(kind='bar', stacked=False, figsize=(11, 6), colormap='Set2')
plt.title('Compare product category sales across regions')
plt.xlabel('Region')
plt.ylabel('Total sales')
plt.xticks(rotation=30)
plt.legend(title='Product category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Save the summaries

In [ ]:
region_sales.to_csv('sales_by_region.csv', index=False)
region_category.to_csv('sales_by_region_and_category.csv')
print('Saved sales_by_region.csv and sales_by_region_and_category.csv')